# NeMo-Skills Competitive Programmer Finetuning

This notebook prepares the OpenCodeReasoning-2 dataset (CPP split) and finetunes a Qwen2.5-Coder-7B-Instruct model. Storage management is optimized to handle large base datasets.

In [1]:
%pip install git+https://github.com/NVIDIA-NeMo/Skills.git
%pip install datasets==2.16.0

  Cloning https://github.com/NVIDIA-NeMo/Skills.git to /tmp/pip-req-build-craszy0a
  Running command git clone --filter=blob:none --quiet https://github.com/NVIDIA-NeMo/Skills.git /tmp/pip-req-build-craszy0a
  Resolved https://github.com/NVIDIA-NeMo/Skills.git to commit acec1b0c6e7614c5a35095e8e192be89b0fa02f7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/NVIDIA/compute-eval.git (to revision e01a5d2) to /tmp/pip-install-z0izgmu4/compute-eval_4fdf23b8432f45a88de3e94ae9672243
  Running command git clone --filter=blob:none --quiet https://github.com/NVIDIA/compute-eval.git /tmp/pip-install-z0izgmu4/compute-eval_4fdf23b8432f45a88de3e94ae9672243
  Running command git checkout -q e01a5d2
  Resolved https://github.com/NVIDIA/compute-eval.git to commit e01a5d2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject

In [ ]:
!ns setup

In [ ]:
import os
from huggingface_hub import snapshot_download
os.environ["NEMO_SKILLS_CONFIGS"] = "/content/cluster_configs"
os.environ["PYTHONPATH"] = "/content/Skills"
os.environ["NVIDIA_API_KEY"] = "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"
MY_TOKEN = "xxx"

In [ ]:
!git pull

In [6]:
%cd /content
!git clone https://github.com/Belaleatsbanana/Skills.git
%cd /content/Skills

/content
Cloning into 'Skills'...
remote: Enumerating objects: 40771, done.
remote: Counting objects: 100% (1915/1915), done.
remote: Compressing objects: 100% (850/850), done.
remote: Total 40771 (delta 1567), reused 1105 (delta 1062), pack-reused 38856 (from 4)
Receiving objects: 100% (40771/40771), 383.27 MiB | 37.17 MiB/s, done.
Resolving deltas: 100% (29621/29621), done.
/content/Skills


In [3]:
ls

cluster_configs/  hf_cache/  opencodereasoning/  sample_data/  Skills/


## Step 1: Data Preparation

Extract questions from the OCR2 dataset. Original solutions are intentionally skipped as we will generate higher quality reasoning traces using a strong model (DeepSeek-R1).

In [9]:
# Questions will be extracted without original solutions to ensure we only train on high-quality reasoning traces
cmd = "python Skills/recipes/opencodereasoning/scripts/prepare_questions.py \
    --output_dir /workspace/opencodereasoning/data"

get_ipython().system(cmd)

python3: can't open file '/content/Skills/Skills/recipes/opencodereasoning/scripts/prepare_questions.py': [Errno 2] No such file or directory


## Step 2: Generate Solutions

Generate synthetic solutions with DeepSeek-R1 via NVIDIA NIM to get high-quality reasoning traces.

In [ ]:
# This uses the 'demo' mode which connects to NVIDIA NIM (requires NVIDIA_API_KEY environment variable)
!python /home/fax/Documents/yappathon/algorithms/gp/NeMo-Skills/recipes/opencodereasoning/pipeline/prepare_solutions.py --mode demo

## Step 3: Prepare SFT Data

Apply formatting for finetuning using the filtered R1 outputs.

In [ ]:
# Pointing to the filtered output from the R1 generation pipeline
input_path = "/workspace/opencodereasoning/data/solution-sdg-toy/filtered/output.jsonl"
!python -m nemo_skills.training.prepare_data \
    ++input_files={input_path} \
    ++output_path=/workspace/opencodereasoning/sft-data.jsonl \
    ++prompt_config=generic/math \
    ++add_unlabeled=true

In [ ]:
# Set this to True if you want to use existing solutions from the base datasets (e.g., apps, taco)
# Set to False if you want to generate synthetic solutions with a separate LLM step
USE_DATASET_SOLUTION = False

cmd = f"python /home/fax/Documents/yappathon/algorithms/gp/NeMo-Skills/recipes/opencodereasoning/scripts/prepare_questions.py \
    --output_dir /workspace/opencodereasoning/data"
if USE_DATASET_SOLUTION:
    cmd += " --use_dataset_solution"

get_ipython().system(cmd)

## Step 2: (Optional) Generate Solutions

If you set `USE_DATASET_SOLUTION = False`, you should run this stage.

In [ ]:
from nemo_skills.pipeline.cli import generate, wrap_arguments
generate(
        input_file="/workspace/opencodereasoning/data/open_code_reasoning_questions.jsonl",
        output_dir="/workspace/opencodereasoning/solutions",
        expname="ocr-solutions",
        cluster="slurm",
        model="Qwen/QwQ-32B-Preview",
        server_type="sglang",
        server_gpus=8,
        prompt_config="/home/fax/Documents/yappathon/algorithms/gp/NeMo-Skills/recipes/opencodereasoning/prompts/generate_cpp_soln.yaml",
        inference=dict(tokens_to_generate=8192, temperature=0.6)
)


## Step 3: Prepare SFT Data

Apply formatting for finetuning.

In [ ]:
input_path = "/workspace/opencodereasoning/data/open_code_reasoning_questions.jsonl" if USE_DATASET_SOLUTION else "/workspace/opencodereasoning/solutions/output.jsonl"
!python -m nemo_skills.training.prepare_data \
    ++input_files={input_path} \
    ++output_path=/workspace/opencodereasoning/sft-data.jsonl \
    ++prompt_config=generic/math \
    ++add_unlabeled=true

## Step 4: Finetune Qwen2.5-Coder-7B-Instruct

Train using NeMo-RL.

In [ ]:
!ns nemo_rl sft \
    --cluster=slurm \
    --training_data=/workspace/opencodereasoning/sft-data.jsonl \
    --hf_model=Qwen/Qwen2.5-Coder-7B-Instruct \
    --backend=megatron \
    --expname=qwen-coder-sft \
    ++policy.max_total_sequence_length=8192 \
    ++sft.max_num_epochs=2